[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kjibran/euramet_ml_noise_workshop/blob/main/00_noise_data_exploration.ipynb)

# Noise Sensors: Exploratory Data Analysis

📡 Three **low-cost** noise nodes (`N1`, `N2`, `N3`, MEMS microphones like the DNMS kit from the hands-on session) were mounted next to a **Class 1** noise
monitoring terminal in a residential street from 1 September to 28 December 2025. All devices report the
A-weighted equivalent level over 10 minutes, `LAeq,10min`, in dB. A nearby weather station adds wind speed and rain.

⚠️ **The data are synthetic.** They were generated to behave like real co-location data, and a few typical faults
are hidden in them. Because the truth is known, we can check how well each method finds them.

## Overview

| File | Content | Time |
|---|---|---|
| `reference_class1.csv` | `laeq_10min` of the Class 1 terminal | UTC |
| `node_N1.csv`, `node_N2.csv`, `node_N3.csv` | `laeq_10min`, `temperature`, `relative_humidity` inside the node | **check it** |
| `meteo.csv` | `wind_speed` [m/s], `rain_mm` per 10 min | UTC |

---

# 🧩 Task 1: What can you say about data quality?

### 📝 Work in your group, note your findings in the provided slide template.

---

# 0 Python Setup and Interface

## 0.0 Notebook Interaction

- **Cells with code**: select the cell and press `Shift`+`Enter` to run it and move on, or `Ctrl`+`Enter` to run it and stay
- **Symbols before code cells**:
  - **▶️ Run**: run the cell as it is
  - **🚀 Action**: run it, change a value, run it again
- \# 💬 **Comment**: Python skips everything after `#`
- Run the cells **from top to bottom**. If something breaks: *Kernel* → *Restart & Run All*.

In [ ]:
# ▶️ Run: get the data (needed on Google Colab, skipped on your own computer)
import os, sys, zipfile
from pathlib import Path

REPO = "euramet_ml_noise_workshop"
REPO_URL = f"https://github.com/kjibran/{REPO}.git"

if not Path("data").exists() and "google.colab" in sys.modules:
    folder = REPO
    if not Path(folder, "data").exists():
        os.system(f"git clone -q {REPO_URL}")
    if not Path(folder, "data").exists():
        from google.colab import files
        print("Download failed. Please upload ml_noise_workshop.zip (from your tutor).")
        zipfile.ZipFile(next(iter(files.upload()))).extractall(".")
        folder = "ml_noise_workshop"
    os.chdir(folder)
print("Data folder found:", Path("data").exists())

In [ ]:
# ▶️ Run
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["date.converter"] = "concise"  # short date labels

## 1 Load the Data

The reference and the weather station use **UTC**. The nodes write their own clock. `energy_mean` is the only
correct way to average levels in dB: it converts to energy, averages, and converts back.

#### ▶️ Run

In [ ]:
# ▶️ Run
FOLDER = Path("./data")
NODES = ["N1", "N2", "N3"]
COLORS = {"N1": "#2a78d6", "N2": "#1baf7a", "N3": "#eb6834", "Reference": "#0b0b0b"}

ref = pl.read_csv(FOLDER / "reference_class1.csv", try_parse_dates=True).rename(
    {"time_utc": "time", "laeq_10min": "ref_laeq"})
meteo = pl.read_csv(FOLDER / "meteo.csv", try_parse_dates=True).rename({"time_utc": "time"})
nodes = pl.concat(
    pl.read_csv(FOLDER / f"node_{node}.csv", try_parse_dates=True).select(pl.lit(node).alias("ID"), pl.all())
    for node in NODES
).sort("ID", "time")


def energy_mean(column: str) -> pl.Expr:
    """Average levels in energy: 10·log10(mean(10^(L/10))). Never take the plain mean of dB values."""
    return 10 * (10 ** (pl.col(column) / 10)).mean().log10()

In [ ]:
# ▶️ Run
print("Reference rows:", ref.height, "| missing reference values:", 17136 - ref.height)
print(nodes.group_by("ID").agg(pl.len().alias("rows"), pl.col("time").min().alias("first"),
                               pl.col("time").max().alias("last")).sort("ID"))

## 2 Data Exploration

### 2.0 📈 Time Series

#### 🚀 Action
- ⏱️ **Averaging window**: try `"10m"`, `"1h"`, `"1d"`, `"1w"`.
- 🧮 **Averaging method**: switch `METHOD` between `"energy"` and `"arithmetic"`. ❓ Which one is lower, and why?

In [ ]:
# 🚀 Action
WINDOW = "1d"
METHOD = "energy"  # 🚀 try "arithmetic"

agg = energy_mean if METHOD == "energy" else (lambda c: pl.col(c).mean())

fig, ax = plt.subplots()
r = ref.sort("time").group_by_dynamic("time", every=WINDOW).agg(agg("ref_laeq").alias("L"))
ax.plot(r["time"], r["L"], color=COLORS["Reference"], lw=2, label="Reference")
for node in NODES:
    d = nodes.filter(pl.col("ID") == node).group_by_dynamic("time", every=WINDOW).agg(agg("laeq_10min").alias("L"))
    ax.plot(d["time"], d["L"], color=COLORS[node], lw=1.2, label=node)
ax.set_ylabel(f"LAeq, {WINDOW} {METHOD} mean (dB)")
ax.legend(ncol=4)
plt.show()

### 2.1 🕐 Daily Profile: Are the Clocks Right?

The median level for each hour of the day, before and after the switch from summer to winter time (26 October).

❓ Which node is shifted, and by how many hours? Does the shift change on 26 October?

#### ▶️ Run

In [ ]:
# ▶️ Run
switch = dt.datetime(2025, 10, 26, 1)
fig, axes = plt.subplots(1, 2, sharey=True)
for ax, (label, cond) in zip(axes, [("before 26 Oct", pl.col("time") < switch), ("after 26 Oct", pl.col("time") >= switch)]):
    r = ref.filter(cond).group_by(pl.col("time").dt.hour().alias("h")).agg(pl.col("ref_laeq").median()).sort("h")
    ax.plot(r["h"], r["ref_laeq"], color=COLORS["Reference"], lw=2.5, label="Reference")
    for node in NODES:
        d = (nodes.filter((pl.col("ID") == node) & cond)
             .group_by(pl.col("time").dt.hour().alias("h")).agg(pl.col("laeq_10min").median()).sort("h"))
        ax.plot(d["h"], d["laeq_10min"], color=COLORS[node], label=node)
    ax.set_title(label); ax.set_xlabel("Hour of day (as written in the file)"); ax.set_xticks(range(0, 24, 3))
axes[0].set_ylabel("Median LAeq,10min (dB)"); axes[0].legend()
plt.show()

### 2.2 🔍 Find the Time Offset

Shift each node by −3 to +3 hours and check how well it then follows the reference (correlation). The best shift
is the clock error.

#### ▶️ Run

In [ ]:
# ▶️ Run
rows = []
for node in NODES:
    d = nodes.filter(pl.col("ID") == node)
    for label, cond in [("before 26 Oct", pl.col("time") < switch), ("after 26 Oct", pl.col("time") >= switch)]:
        best = max(
            ((lag, d.with_columns(pl.col("time") - pl.duration(hours=lag)).join(ref, on="time")
              .filter(cond).select(pl.corr("laeq_10min", "ref_laeq")).item()) for lag in range(-3, 4)),
            key=lambda x: x[1])
        rows.append({"node": node, "period": label, "best shift (h)": best[0], "correlation": round(best[1], 3)})
pl.DataFrame(rows)

### 2.3 🛠️ Fix the Clock

A node that writes local time must be converted to UTC before any comparison. The function below does that for `N1`.
**Every correction needs a reason**: here, the shift is +2 h in summer time and +1 h in winter time.

#### ▶️ Run

In [ ]:
# ▶️ Run

def fix_clock(nodes: pl.DataFrame) -> pl.DataFrame:
    """N1 logs local time in Bern (UTC+2 in summer, UTC+1 after 26 Oct). Convert it to UTC like the others.
    The repeated hour on 26 Oct cannot be resolved, so those readings are dropped."""
    n1 = nodes.filter(pl.col("ID") == "N1").with_columns(
        pl.col("time").dt.replace_time_zone("Europe/Zurich", ambiguous="null")
        .dt.convert_time_zone("UTC").dt.replace_time_zone(None)
    ).drop_nulls("time")
    return pl.concat([n1, nodes.filter(pl.col("ID") != "N1")]).sort("ID", "time")

def build_table(nodes: pl.DataFrame) -> pl.DataFrame:
    """One row per node and 10 minutes: node reading, reference, weather. Rows without a reference value are dropped."""
    return (
        fix_clock(nodes)
        .join(ref, on="time", how="left")
        .join(meteo, on="time", how="left")
        .drop_nulls(["laeq_10min", "ref_laeq"])
        .with_columns(
            hour=pl.col("time").dt.hour(),
            day=(pl.col("time") - pl.datetime(2025, 9, 1)).dt.total_minutes() / 1440,
        )
    )

data = build_table(nodes)
print(data.group_by("ID").agg(pl.len().alias("rows with reference")).sort("ID"))

### 2.4 📉 Node Against Reference

Each dot is one 10-minute period. Colour the dots by another quantity.

#### 🚀 Action
- 📡 `SENSOR`: `"N1"`, `"N2"`, `"N3"`
- 🌬️ `FEATURE`: `"wind_speed"`, `"rain_mm"`, `"temperature"`, `"hour"`, `"day"`

❓ Where does the node leave the 1:1 line? At low levels? In wind? In rain? Later in the period?

In [ ]:
# 🚀 Action
SENSOR = "N1"
FEATURE = "wind_speed"

d = data.filter(pl.col("ID") == SENSOR)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sc = ax.scatter(d["ref_laeq"], d["laeq_10min"], c=d[FEATURE], s=4, cmap="viridis", alpha=0.6)
ax.plot([30, 75], [30, 75], "k--", lw=1)
ax.set_xlabel("Reference LAeq,10min (dB)"); ax.set_ylabel(f"{SENSOR} LAeq,10min (dB)")
plt.colorbar(sc, label=FEATURE)
plt.show()

### 2.5 📆 Stability Over Time

The daily median of (node − reference), using only dry periods with little wind and daytime levels above 50 dB,
so that wind, rain and the noise floor do not hide the instrument itself.

❓ Which node stays stable? Which one changes slowly? Which one changes suddenly, and on which day?

#### ▶️ Run

In [ ]:
# ▶️ Run
clean = data.filter((pl.col("rain_mm") == 0) & (pl.col("wind_speed") < 4) & (pl.col("ref_laeq") > 50))
daily = (clean.with_columns((pl.col("laeq_10min") - pl.col("ref_laeq")).alias("diff"))
         .group_by("ID", pl.col("time").dt.date().alias("date")).agg(pl.col("diff").median()).sort("ID", "date"))
fig, ax = plt.subplots()
for node in NODES:
    d = daily.filter(pl.col("ID") == node)
    ax.plot(d["date"], d["diff"], ".-", color=COLORS[node], label=node, lw=1)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Daily median of node − reference (dB)"); ax.legend()
plt.show()

## 📝 Summarise your data quality observations in the template

- Clock and time stamps
- Gaps
- Low levels (noise floor)
- Wind and rain
- Stability over time